#### 1. Setup
Use 'elt' kernel / environment.  
Run shell commands in the olist_dbt directory; not the 'notebooks' directory. 

This guide closely mirrors the steps in:   
https://docs.elementary-data.com/oss/quickstart/quickstart-cli-package  

#### 2. Add elementary to `packages.yml`
```
packages:
  - package: elementary-data/elementary
    version: 0.25.1
```

#### 3. Add elementary model to your `dbt_project.yml`. Import the package.
```
models:
  elementary:
    +schema: "elementary"
```

Run `dbt deps`.

#### 4. Build Elementary models
Run `dbt run --select elementary`.   
This will create dataset `olist_dev_elementary`; and mostly empty tables, that will be updated with artifacts, metrics and test results in your future dbt executions.

#### 5. Run tests
Run `dbt run`.  
`elementary_test_results` table should now be populated.

#### 6. Get Elementary Connection Profile 
To generate the profile, run:   
`dbt run-operation elementary.generate_elementary_cli_profile`  

We get an output similar to the following:   
```
elementary:
  outputs:
    default:
      type: "bigquery"
      project: "project-7781a5d3-3f4e-4cd5-a43"
      dataset: "olist_dev_elementary"
      method: "<AUTH_METHOD>"  # Configure your auth method and add the required fields according to https://docs.getdbt.com/reference/warehouse-setups/bigquery-setup#authentication-methods
      threads: 1
```

Copy this into `profiles.yml`.  

#### 6. Build the core model, with fact and dimension tables.
a. Prune columns that are not of interest to us.   
b. Add column `order_total_price` in `fct_orders` table. This sums up all the order line items' prices (excludes freight).
c. Add column `latest_review` in `fct_order_reviews` table. This examines `review_answer_timestamp` and labels the most recent review as True.

`dbt build --select core --profiles-dir .`   

#### 7. Build the datamart model, only with `delivered` orders.

`dbt build --select datamart --profiles-dir .`   

#### P.S. Targets can also be run, test, build individually or incrementally, e.g.

`dbt test --select fct_orders.sql --profiles-dir . --store-failures`   

#### 8. Generate documentation.

`dbt docs generate --select olist_dbt --profiles-dir . --static`

Move the files to the docs directory  
`cp -r target/* ../docs`

Add, commit, and push  
`git add ../docs/*`  
`git commit -m "Update dbt documentation"`  
`git push origin`

The documentation can be viewed at:   
https://cheongnicole.github.io/DS6-Module-2-Project-Group-6/